In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

filename = next(iter(uploaded))
df = pd.read_csv(filename)

print(df.shape)
df.head()

Saving hotel_bookings_clean.csv to hotel_bookings_clean.csv
(119205, 34)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_nights,total_guests,previous_bookings,room_changed,season
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,Transient,0.0,0,0,0,2,0,0,Summer
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,Transient,0.0,0,0,0,2,0,0,Summer
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,Transient,75.0,0,0,1,1,0,1,Summer
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,Transient,75.0,0,0,1,1,0,0,Summer
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,Transient,98.0,0,1,2,2,0,0,Summer


Load libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Target variable: 0 = No cancellation, 1 = Cancellation
target = "is_canceled"

candidate_features = [
    "hotel",
    "lead_time",
    "adults",
    "children",
    "babies",
    "meal",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "customer_type",
    "stays_in_week_nights",
    "stays_in_weekend_nights",
    "total_of_special_requests",
    "previous_cancellations"
]

features = [c for c in candidate_features if c in df.columns]

data = df[features + [target]].copy()
data[target] = pd.to_numeric(data[target], errors="coerce")
data = data.dropna(subset=[target])
data = data[data[target].isin([0, 1])]
data[target] = data[target].astype(int)

X = data[features]
y = data[target]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# 80% training, 20% testing; preserve target proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Number of predictors:", len(features))
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).rename("Proportion"))

2. Evaluate trees with varying maximum depths

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

param_grid = {
    "classifier__max_depth": [3, 5, 7, 10, 15],
    "classifier__min_samples_leaf": [1, 5, 10]
}

scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1-score": "f1",
    "ROC-AUC": "roc_auc"
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=scoring,
    refit="F1-score",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV F1-score:", round(grid_search.best_score_, 4))

best_depth = grid_search.best_params_["classifier__max_depth"]
best_leaf = grid_search.best_params_["classifier__min_samples_leaf"]
final_model = grid_search.best_estimator_

Visualize comparison

In [ ]:
# Keep one row per depth, using the best leaf size
cv_df = pd.DataFrame(grid_search.cv_results_)
best_leaf = grid_search.best_params_["classifier__min_samples_leaf"]

depth_results = (
    cv_df[cv_df["param_classifier__min_samples_leaf"] == best_leaf]
    .rename(columns={
        "param_classifier__max_depth": "Max Depth",
        "mean_test_Accuracy": "Accuracy",
        "mean_test_Precision": "Precision",
        "mean_test_Recall": "Recall",
        "mean_test_F1-score": "F1-score",
        "mean_test_ROC-AUC": "ROC-AUC",
    })
    [["Max Depth", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]]
    .astype({"Max Depth": int})
    .sort_values("Max Depth")
)

display(depth_results.round(4))

plt.figure(figsize=(11, 6))

for metric in ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]:
    plt.plot(
        depth_results["Max Depth"],
        depth_results[metric],
        marker="o",
        label=metric
    )

plt.xlabel("Maximum Tree Depth")
plt.ylabel("Cross-validation score")
plt.title("Decision Tree Performance by Maximum Depth (5-fold CV)")
plt.xticks(depth_results["Max Depth"])
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

3. Evaluate trees with varying minimum leaf sizes

In [ ]:
cv_df = pd.DataFrame(grid_search.cv_results_)
best_depth = grid_search.best_params_["classifier__max_depth"]

leaf_results = (
    cv_df[cv_df["param_classifier__max_depth"] == best_depth]
    .rename(columns={
        "param_classifier__min_samples_leaf": "Min Samples Leaf",
        "mean_test_Accuracy": "Accuracy",
        "mean_test_Precision": "Precision",
        "mean_test_Recall": "Recall",
        "mean_test_F1-score": "F1-score",
        "mean_test_ROC-AUC": "ROC-AUC",
    })
    [["Min Samples Leaf", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]]
    .astype({"Min Samples Leaf": int})
    .sort_values("Min Samples Leaf")
)

display(leaf_results.round(4))

4. Evaluate the selected decision tree

In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=best_depth,
        min_samples_leaf=best_leaf,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob)
    ]
})

print(f"Selected max_depth: {best_depth}")
print(f"Selected min_samples_leaf: {best_leaf}")
display(final_metrics.round(4))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No cancellation", "Cancellation"],
    zero_division=0
))

5. Visualize the final decision tree

In [ ]:
fitted_preprocessor = final_model.named_steps["preprocessor"]
tree_classifier = final_model.named_steps["classifier"]

transformed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

plt.figure(figsize=(24, 10))

plot_tree(
    tree_classifier,
    feature_names=transformed_feature_names,
    class_names=["No cancellation", "Cancellation"],
    filled=True,
    rounded=True,
    max_depth=4,
    fontsize=8
)

plt.title("Selected Decision Tree (First Four Levels)")
plt.tight_layout()
plt.show()

Feature importance


In [ ]:
# Get feature importances from the trained decision tree
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
feature_values = final_model.named_steps["classifier"].feature_importances_

# Create a DataFrame
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_values
})

# Select the 15 most important features
importance = importance.sort_values(
    by="Importance", ascending=False
).head(10)

# Plot
plt.figure(figsize=(10, 7))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Top 10 Decision Tree Feature Importances")
plt.tight_layout()
plt.show()

6. Extract the decision rules

In [ ]:
from sklearn.tree import export_text

# Get the trained decision tree from your pipeline
preprocessor = final_model.named_steps["preprocessor"]
tree_model = final_model.named_steps["classifier"]

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Print the IF-THEN rules
rules_text = export_text(
    tree_model,
    feature_names=list(feature_names),
    max_depth=5
)

print("DECISION RULES")
print(rules_text)

7. Classify bookings using the rules

In [ ]:
import numpy as np
from sklearn.tree import _tree

# Transform test data using the same fitted preprocessing pipeline
X_test_transformed = preprocessor.transform(X_test)

# Convert sparse matrix to array if necessary
if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

tree = tree_model.tree_

def predict_from_rules(X):
    predictions = []

    for sample in X:
        node = 0

        # Follow the IF-THEN conditions to a leaf
        while tree.feature[node] != _tree.TREE_UNDEFINED:
            feature_index = tree.feature[node]
            threshold = tree.threshold[node]

            if sample[feature_index] <= threshold:
                node = tree.children_left[node]
            else:
                node = tree.children_right[node]

        # The leaf's majority class is the prediction
        class_index = np.argmax(tree.value[node][0])
        prediction = tree_model.classes_[class_index]
        predictions.append(prediction)

    return np.array(predictions)


# Predictions from both approaches
y_pred_tree = final_model.predict(X_test)
y_pred_rules = predict_from_rules(X_test_transformed)

print("Tree predictions:", y_pred_tree[:10])
print("Rule predictions:", y_pred_rules[:10])

print(
    "Predictions identical:",
    np.array_equal(y_pred_tree, y_pred_rules)
)

8. Compare the two classifiers

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

def evaluate_model(y_true, y_pred, name):
    return {
        "Classifier": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_true, y_pred, zero_division=0
        )
    }

comparison = pd.DataFrame([
    evaluate_model(y_test, y_pred_tree, "Decision Tree"),
    evaluate_model(y_test, y_pred_rules, "Decision Rules")
])

display(comparison.round(4))

print("Decision Tree Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tree))

print("\nDecision Rules Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rules))

9. Visualize the comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

comparison_long = comparison.melt(
    id_vars="Classifier",
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=comparison_long,
    x="Metric",
    y="Score",
    hue="Classifier"
)
plt.ylim(0, 1)
plt.title("Decision Tree vs Decision Rule-Based Classifier")
plt.tight_layout()
plt.show()